# Armenian MT: Ashkharabar → Grabar

Train a Transformer Encoder-Decoder to translate Modern Armenian to Classical Armenian using Kaggle GPU T4.

In [ ]:
!pip install -q sentencepiece tokenizers pandas pyarrow
import os
os.makedirs('/kaggle/working/data', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/tokenizer', exist_ok=True)
print('Directories created')

In [ ]:
# Download Classical Armenian dataset
!wget -q 'https://huggingface.co/datasets/catherinearnett/classical_armenian_pd/resolve/main/classical_armenian_pd.parquet' -O /kaggle/working/data/classical_armenian_pd.parquet
!ls -la /kaggle/working/data/

In [ ]:
# Download Modern Armenian sentences dataset
!wget -q 'https://huggingface.co/datasets/EdUarD0110/Armenian_sentences_100000/resolve/main/data/hag_translated.csv' -O /kaggle/working/data/armenian_sentences_100k.csv
!ls -la /kaggle/working/data/

In [ ]:
import pandas as pd
import re
import random
from pathlib import Path

random.seed(42)

def split_sentences(text):
    if not text or not isinstance(text, str):
        return []
    text = text.strip()
    sentences = re.split(r'[:։.!?]', text)
    sentences = [s.strip() for s in sentences if s.strip() and len(s.strip()) > 8]
    return sentences

print('Loading Classical Armenian...')
df_cl = pd.read_parquet('/kaggle/working/data/classical_armenian_pd.parquet')
cl_sents = []
for _, row in df_classical.iterrows():
    content = row.get('content', '')
    if isinstance(content, str):
        cl_sents.extend(split_sentences(content))
print(f'Classical sentences: {len(cl_sents)}')

print('Loading Modern Armenian...')
df_mod = pd.read_csv('/kaggle/working/data/armenian_sentences_100k.csv')
mod_sents = [s.strip() for s in df_mod['armenian_translation'].tolist() if isinstance(s, str) and s.strip() and len(s.strip()) > 8]
print(f'Modern sentences: {len(mod_sents)}')

import bisect
mod_with_len = sorted([(len(s), s) for s in mod_sents])
mod_lengths = [x[0] for x in mod_with_len]

pairs = []
used = set()
cl_with_len = sorted([(len(s), s) for s in cl_sents])

for cl_len, cs in cl_with_len:
    if len(pairs) >= 3000:
        break
    pos = bisect.bisect_left(mod_lengths, cl_len)
    best_idx = None
    best_diff = float('inf')
    for off in range(-5, 6):
        idx = pos + off
        if 0 <= idx < len(mod_with_len) and idx not in used:
            diff = abs(mod_with_len[idx][0] - cl_len)
            if diff < best_diff:
                best_diff = diff
                best_idx = idx
    if best_idx is not None and best_diff < 30:
        pairs.append((mod_with_len[best_idx][1], cs))
        used.add(best_idx)

print(f'Parallel pairs created: {len(pairs)}')

with open('/kaggle/working/data/parallel_pairs.txt', 'w', encoding='utf-8') as f:
    for m, c in pairs:
        f.write(f'{m}\t{c}\n')

print(f'Saved {len(pairs)} pairs to /kaggle/working/data/parallel_pairs.txt')

In [ ]:
import torch, torch.nn as nn, math
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 32
print(f'Device: {DEVICE}')

class PDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=64):
        self.pairs = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    self.pairs.append(parts)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        s_ids = [1] + self.tok.encode(src).ids + [2]
        t_ids = [1] + self.tok.encode(tgt).ids + [2]
        s_ids = s_ids[:self.max_len]
        t_ids = t_ids[:self.max_len]
        return torch.tensor(s_ids), torch.tensor(t_ids)

def collate(batch):
    srcs, tgts = zip(*batch)
    src = torch.nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=3)
    tgt = torch.nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=3)
    return src, tgt

# Build tokenizer
tok = Tokenizer(models.BPE(unk_token='[UNK]'))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=4000, special_tokens=['[UNK]', '[BOS]', '[EOS]', '[PAD]'], min_frequency=1)
all_text = []
with open('/kaggle/working/data/parallel_pairs.txt', 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            all_t.extend(parts)
tok.train_from_iterator(all_t, tr)
tok.save('/kaggle/working/tokenizer/sp_model.json')
print(f'Vocab size: {tok.get_vocab_size()}')

ds = PDataset('/kaggle/working/data/parallel_pairs.txt', tok)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

class PE(nn.Module):
    def __init__(self, d, drop=0.1, ml=5000):
        super().__init__()
        self.dropout = nn.Dropout(drop)
        pe = torch.zeros(ml, d)
        pos = torch.arange(0, ml).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TM(nn.Module):
    def __init__(self, vocab, d=256, nh=8, nl=4, ff=512, drop=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pe = PE(d, dropout)
        self.trans = nn.Transformer(d, nh, nl, nl, ff, drop, batch_first=True)
        self.gen = nn.Linear(d, v)
        self.d = d

    def forward(self, s, t):
        s_e = self.pe(self.emb(s) * math.sqrt(self.d))
        t_e = self.pe(self.emb(t) * math.sqrt(self.d))
        out = self.trans(s_e, t_e,
            src_key_padding_mask=(s==3),
            tgt_mask=torch.tril(torch.ones(t.size(1), t.size(1))).bool().to(s.device),
            tgt_key_padding_mask=(t==3),
            memory_key_padding_mask=(s==3))
        return self.gen(out)

model = TM(tok.get_vocab_size()).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
crit = nn.CrossEntropyLoss(ignore_index=3)

for ep in range(10):
    model.train()
    tl = 0
    for s, t in dl:
        s, t = s.to(DEVICE), t.to(DEVICE)
        ti, to = t[:, :-1], t[:, 1:]
        lo = model(s, ti)
        loss = crit(lo.reshape(-1, lo.size(-1)), to.reshape(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item()
    print(f'Epoch {ep+1}/10: loss={tl/len(dl):.4f}')

torch.save({'model': model.state_dict(), 'vocab': tok.get_vocab_size()}, '/kaggle/working/checkpoints/armenian_mt.pt')
print('Checkpoint saved!')

In [ ]:
!cp /kaggle/working/tokenizer/sp_model.json /kaggle/working/tokenizer.json
!ls -la /kaggle/working/checkpoints/
print('Download checkpoint from Output panel')